<a href="https://colab.research.google.com/github/asoto59g/Meteorito-Yucat-n/blob/master/chicxulub_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌋 Detección de Cenotes — Cráter Chicxulub
**Sentinel-1 SAR | Google Earth Engine + Google Colab**

Este notebook detecta cenotes en la zona del anillo del cráter de Chicxulub (Yucatán, México) utilizando datos de radar SAR de Sentinel-1.

## 0. Instalación y Autenticación

In [ ]:
# Instalar geemap (incluye earthengine-api)
# Solo se necesita ejecutar una vez por sesión de Colab
!pip install -q geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.3 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

# Autenticación — se abrirá un enlace para autorizar
ee.Authenticate()

# Inicializar Earth Engine
ee.Initialize(project='ee-oasotob')  # ← Reemplaza con tu proyecto GEE

## 1. Área y Geometría

In [ ]:
# ==============================
# 1. ÁREA Y GEOMETRÍA
# ==============================
roi = ee.Geometry.Rectangle([-91.5, 19.5, -87.5, 22.5])

center = ee.Geometry.Point([-89.6, 21.3])

outer = center.buffer(90000)   # 90 km
inner = center.buffer(80000)   # 80 km
ring = outer.difference(inner)

## 2. Carga Sentinel-1

In [ ]:
# ==============================
# 2. CARGA SENTINEL-1
# ==============================
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(roi)
      .filterDate('2023-01-01', '2023-12-31')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
      .select('VV'))

# Composición (mediana reduce speckle)
s1_median = s1.median().clip(roi)

## 3. Suavizado (Reducir Speckle)

In [ ]:
# ==============================
# 3. SUAVIZADO (REDUCIR SPECKLE)
# ==============================
smooth = s1_median.focal_mean(30, 'circle', 'meters')

## 4. Detección de Agua (SAR)

In [ ]:
# ==============================
# 4. DETECCIÓN DE AGUA (SAR)
# ==============================
# Agua = valores muy bajos (oscuro)
# Ajusta este umbral si es necesario
water = smooth.lt(-17)

## 5. Limpieza y Filtro Tierra

In [ ]:
# ==============================
# 5. LIMPIEZA Y FILTRO TIERRA
# ==============================
waterClean = water.focal_max(1)

# Máscara de tierra (México) para excluir el mar
mexico = (ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
          .filter(ee.Filter.eq('country_na', 'Mexico')))

# Agua solo en tierra
waterLand = waterClean.updateMask(waterClean).clip(mexico)

## 5b. Vectorización (Para Visibilidad Permanente)

In [ ]:
# ==============================
# 5b. VECTORIZACIÓN
# ==============================
# Convertir la imagen de agua (solo tierra) a vectores (polígonos)
waterVectors = waterLand.reduceToVectors(**{
    'geometry': roi,
    'scale': 30,
    'geometryType': 'polygon',
    'eightConnected': True,
    'labelProperty': 'water',
    'reducer': ee.Reducer.countEvery(),
    'maxPixels': 1e9,
    'bestEffort': True
})

# Filtrar polígonos muy pequeños (ruido del radar)
# Un cenote debe tener al menos ~1500 m² (aprox. 4 píxeles de 20m)
waterFiltered = waterVectors.filter(ee.Filter.gt('count', 3))

# Buffer de 1000 metros (1 km) alrededor de cada cenote
waterBufferVector = waterFiltered.map(lambda f: f.buffer(1000))

## 6. Bordes (Opcional — Canny Edge)

In [ ]:
# ==============================
# 6. BORDES (OPCIONAL)
# ==============================
edges = ee.Algorithms.CannyEdgeDetector(**{
    'image': smooth,
    'threshold': 1,
    'sigma': 1
})

## 7. Cenotes en el Anillo del Cráter

In [ ]:
# ==============================
# 7. CENOTES EN EL ANILLO (TIERRA)
# ==============================
cenotesRing = waterLand.clip(ring)

## 7b. Visibilidad Permanente (Trucos de Renderizado)

In [ ]:
# ==============================
# 7b. VISIBILIDAD PERMANENTE
# ==============================

# 1. Cenotes como puntos con tamaño fijo en pantalla
cenotePoints = waterFiltered.map(lambda f: f.centroid())

# "Dibujar" los puntos con un radio de 4 píxeles constante
waterVisDots = cenotePoints.draw(**{
    'color': '0000FF',
    'pointRadius': 4
})

# 2. Buffer de 1000m visible con grosor de 2 píxeles
waterBufferVis = (waterBufferVector
                  .style(**{
                      'color': '00FFFF66',
                      'fillColor': '00FFFF22',
                      'width': 1
                  })
                  .focal_max(2, 'square', 'pixels'))

## 8. Visualización en Mapa Interactivo

In [ ]:
# ==============================
# 8. VISUALIZACIÓN
# ==============================

# Crear mapa interactivo con geemap
Map = geemap.Map()

# Fondo: Google Satellite Hybrid
Map.setOptions('HYBRID')

# Centrar en el cráter de Chicxulub
Map.centerObject(center, 8)

# --- Capas ---

# SAR Sentinel-1 (mediana)
Map.addLayer(s1_median, {'min': -25, 'max': 0}, 'Sentinel-1 VV (mediana)', False)

# Imagen suavizada
Map.addLayer(smooth, {'min': -25, 'max': 0}, 'SAR Suavizado', False)

# Máscara de agua en tierra
Map.addLayer(waterLand, {'palette': ['blue']}, 'Agua en Tierra', False)

# Bordes Canny
Map.addLayer(edges, {'min': 0, 'max': 1, 'palette': ['white']}, 'Bordes Canny', False)

# Cenotes en el anillo del cráter
Map.addLayer(cenotesRing, {'palette': ['cyan']}, 'Cenotes en Anillo', False)

# Cenotes Detectados — Puntos Azules (capa principal)
Map.addLayer(waterVisDots, {}, 'Cenotes Chicxulub')

# Anillo del cráter (contorno)
Map.addLayer(ring, {'color': 'FF000088'}, 'Anillo Chicxulub', True)

# Mostrar mapa
Map

Map(center=[21.3, -89.6], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

## 9. Debug / Información

In [ ]:
# ==============================
# DEBUG
# ==============================
print('Sentinel-1 colección — imágenes:', s1.size().getInfo())
print('Banda SAR mediana — bandas:', s1_median.bandNames().getInfo())
print('Cenotes detectados (polígonos):', waterFiltered.size().getInfo())
print('Cenotes en anillo — bandas:', cenotesRing.bandNames().getInfo())

Sentinel-1 colección — imágenes: 106
Banda SAR mediana — bandas: ['VV']
Cenotes detectados (polígonos): 3363
Cenotes en anillo — bandas: ['VV']


## 10. Exportar Cenotes a Google Drive (Opcional)